In [1]:
import polars as pl
from pathlib import Path
import jellyfish
from nltk.corpus import names
import nltk
#nltk.download('names')
import re
import altair as alt

In [2]:
samples_df = pl.read_parquet("jails_pdfs_full.parquet")

In [3]:
yes_count = (samples_df['Resulting Death?'] == 'Yes').sum()


print (yes_count)

137


In [4]:
entries = ["Restraints Used\n", "Smattery"]


def clean_occurrences(entries):

    actual_occurrences = ["Suicide (method)", "Suicide (attempt)", "Homicide", "Homicide Attempt", "Escape", "Escape Attempt",
                "Fire", "Serious Injury", "Battery", "Riot of Rebellion", "Sex Offense", "Assault on Staff",
                "Assault among Detainees", "Fighting among Detainees", "Restraints Used", "OC Spray Used", "Other (specify)"]
    
    if entries.is_empty():
        return ["Error, no occurrence found"]

    cleaned_list = []
    for term in entries:
        best_choice = None
        highest_jaro = 0.0
        for compare_term in actual_occurrences:
            jaro_score = jellyfish.jaro_similarity(term, compare_term)
            if jaro_score > highest_jaro:
                highest_jaro = jaro_score
                best_choice = compare_term

        if best_choice in ["Suicide (method)", "Suicide (attempt)", "Other (specify)"]:
            # Keep the original term (possibly has colon text, like "Other (specify): Fight")
            cleaned_list.append(term)
        else:
            if best_choice not in cleaned_list:
                cleaned_list.append(best_choice)

    return cleaned_list

In [5]:
all_names = set(names.words())

test_list = ["Deaunte Campbell","3/12/2002","10/13/2021"]

def get_inmate_names(table_list):
    inmate_list = []
    pattern_punctuation = r"[^a-zA-Z0-9/ ]"
    for item in table_list:
        cleaned_item = re.sub(pattern_punctuation, '', item)
        tokens = cleaned_item.strip().split()
        for token in tokens:
            token = token.capitalize()
            if token in all_names:
                inmate_list.append(cleaned_item)
                break
    return inmate_list

get_inmate_names(test_list)

[]

In [6]:
occurrence_check = samples_df.select(pl.col("Occurrence").map_elements(clean_occurrences).alias("Cleaned Occurrences"))

C:\Users\ganon\AppData\Local\Temp\ipykernel_20508\2675670731.py:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  occurrence_check = samples_df.select(pl.col("Occurrence").map_elements(clean_occurrences).alias("Cleaned Occurrences"))


In [7]:
samples_df = samples_df.with_columns(pl.col("Occurrence").map_elements(clean_occurrences).alias("Cleaned Occurrences"))

C:\Users\ganon\AppData\Local\Temp\ipykernel_20508\3155367749.py:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  samples_df = samples_df.with_columns(pl.col("Occurrence").map_elements(clean_occurrences).alias("Cleaned Occurrences"))


In [8]:
samples_df = samples_df.with_columns(pl.col("Table Contents").map_elements(get_inmate_names).alias("Inmate Names"))

C:\Users\ganon\AppData\Local\Temp\ipykernel_20508\3131983131.py:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  samples_df = samples_df.with_columns(pl.col("Table Contents").map_elements(get_inmate_names).alias("Inmate Names"))


In [9]:
# Clean facility name:
def clean_facility_name(df):
    """Clean facility names using Polars expressions"""
    return df.with_columns(
        pl.col('Facility Name')
        # First, clean OCR artifacts
        .str.replace_all(r'\\n|\\\\|\\/', ' ') 
        .str.replace_all(r'\n', ' ')
        .str.replace_all(r'[\\\/]+', ' ')
        .str.replace_all(r'\s+', ' ') 
        .str.strip_chars()
        # Extract facility name after any variation of "Name:"
        .str.extract(r'cility Name:\s*(.+)', 1)
        # Additional cleanup of extracted names
        .str.replace_all(r'[^a-zA-Z\s\'-]', ' ')  # Keep only letters, spaces, apostrophes, hyphens
        .str.replace_all(r'\s+', ' ')  # Clean multiple spaces again
        .str.strip_chars()  # Final trim
        # Stop at any word ending in "ddres" and remove that word + everything after
        .str.replace_all(r'\s+(ddress|ddrace).*$', '')
        .str.replace_all(r"(?i)\bSheriff's\b", "Sheriffs")
        .str.to_titlecase()
        .alias('Cleaned Facility Name')
    )

samples_df = clean_facility_name(samples_df)

# Date cleanning
def clean_date_occurrence(df):
    return df.with_columns(
        pl.col("Date")
          # OCR cleanup (keep forward slashes)
          .str.replace_all(r"\\n|\\\\", " ")
          .str.replace_all(r"\n", " ")
          .str.replace_all(r"\s+", " ")
          .str.strip_chars()
          # Extract the MM/DD/YYYY text
          .str.extract(r"Date of Occurrence:\s*([0-9]{1,2}/[0-9]{1,2}/(20[0-9]{2}))", 1)
          # Parse & re‐format to zero‐padded MM/DD/YYYY
          .str.strptime(pl.Date, "%m/%d/%Y", strict=False)
          .alias("temp_date")
    ).with_columns(
        # Only keep dates between 2000 and 2025, otherwise set to null
        pl.when(pl.col("temp_date").dt.year().is_between(2000, 2025))
        .then(pl.col("temp_date").dt.strftime("%m/%d/%Y"))
        .otherwise(None)
        .alias("Cleaned Date")
    ).drop("temp_date")

samples_df = clean_date_occurrence(samples_df)

In [10]:
samples_df = samples_df.drop(["OCR_Confidence", "OCR_Word_Count", "OCR_Text_Detected", "Occurrence", "Facility Name", "Date", "Cook County?"])

In [11]:
samples_df = samples_df.with_columns(pl.col("Cleaned Occurrences").list.join(",").alias("Cleaned Occurrences"))

samples_df = samples_df.with_columns(pl.col("Inmate Names").list.join(",").alias("Inmate Names"))

samples_df = samples_df.with_columns(pl.col("Table Contents").list.join(",").alias("Table Contents"))

In [12]:
samples_df = samples_df.select(["Report ID", "Facility Type", "Cleaned Facility Name", "Address", "Phone Number", "Cleaned Date", "Time of Day", "AM or PM", "Table Contents", "Inmate Names", "Cleaned Occurrences", "Injuries?", "Resulting Death?", 'Deceased Cause, Date, and Time', 'Deceased on Suicide Watch', 'Deceased Reporter', 'Deceased Examined by Physician', 'Deceased Signs of Illness'])

In [ ]:
samples_df

In [13]:
samples_df = samples_df.with_columns(
    pl.when(pl.col("Cleaned Occurrences") == "")
    .then(pl.lit("Error, no occurrence found"))
    .otherwise(pl.col("Cleaned Occurrences"))
    .alias("Cleaned Occurrences")
)

def clean_other(df):
    pattern = r"specify"
    df = df.with_columns(
    pl.when(pl.col("Cleaned Occurrences").str.contains(pattern))
    .then(pl.col("Cleaned Occurrences").str.extract(r"specify(.*)", group_index=1))
    .otherwise(pl.col("Cleaned Occurrences"))
    .alias("Cleaned Occurrences")
    )

    return df

total_occurrence_result = (
    samples_df
    .with_columns(
        pl.col("Cleaned Occurrences").str.split(",").alias("Cleaned Occurrences")
    )
    .explode("Cleaned Occurrences")
    .filter(pl.col("Cleaned Occurrences").is_not_null())
    .select(
        pl.col("Cleaned Occurrences").value_counts(sort=True)
    )
    .unnest("Cleaned Occurrences")
    .head(12)
)

total_occurrence_result = clean_other(total_occurrence_result)

total_occurrence_result = total_occurrence_result.filter(~pl.col("Cleaned Occurrences").is_in(["Error", " no occurrence found"]))

total_occurrence_chart = alt.Chart(total_occurrence_result).mark_bar().encode(
    x=alt.X("count:Q", title="Count"),
    y=alt.Y("Cleaned Occurrences:N", title="Occurrence").sort('-x')
).properties(
    title="Most Common Occurrences Across All Jails"
)

# Footnote as a separate text chart
footnote = alt.Chart(pl.DataFrame({
    "text": ["Note: Occurrences may be concurrent. Missing 3,418 Occurrences (12.5%)"],
    "x": [0],
    "y": [0]
})).mark_text(
    align="left",
    baseline="bottom",
    fontSize=10,
    dx=0,
    dy=20,  # distance below the main chart area
    color="gray"
).encode(
    x=alt.value(-50),
    y=alt.value(230),  # just below the chart (chart height + some offset)
    text="text:N"
)

# Use layering to draw the footnote over/under the chart
total_occurrence_chart = alt.layer(
    total_occurrence_chart,
    footnote
).configure_view(
    stroke=None  # removes box around the chart
)

total_occurrence_chart

alt.LayerChart(...)

In [14]:
top_prisons = (
    samples_df.select(
        pl.col("Cleaned Facility Name").value_counts(sort=True)
    )
    .unnest("Cleaned Facility Name")
    .head(11)
    )

top_prisons = top_prisons.filter(pl.col("Cleaned Facility Name") != "")

# Footnote as a separate text chart
footnote = alt.Chart(pl.DataFrame({
    "text": ["Note: Missing 2,239 County Names (8.19%)"],
    "x": [0],
    "y": [0]
})).mark_text(
    align="left",
    baseline="bottom",
    fontSize=10,
    dx=0,
    dy=20,  # distance below the main chart area
    color="gray"
).encode(
    x=alt.value(-50),
    y=alt.value(230),  # just below the chart (chart height + some offset)
    text="text:N"
)

top_prisons_chart = alt.Chart(top_prisons).mark_bar().encode(
    x=alt.X("count:Q", title="Count"),
    y=alt.Y("Cleaned Facility Name:N", title="Jail").sort('-x')
).properties(
    title="Occurrence Counts by Jails"
)

# Use layering to draw the footnote over/under the chart
top_prisons_chart = alt.layer(
    top_prisons_chart,
    footnote
).configure_view(
    stroke=None  # removes box around the chart
)

top_prisons_chart

alt.LayerChart(...)

In [15]:
pattern = r"\bspecify\b"

other_occurrence_table = samples_df.filter(
    pl.col("Cleaned Occurrences").str.contains(pattern)
)
    
other_occurrence_table = clean_other(other_occurrence_table)

other_occurrence_table = (
    other_occurrence_table.select(
        pl.col("Cleaned Occurrences").value_counts(sort=True)
    )
    .unnest("Cleaned Occurrences")
    .head(10)
)

other_occurrence_chart = alt.Chart(other_occurrence_table).mark_bar().encode(
    x=alt.X("count:Q", title="Count"),
    y=alt.Y("Cleaned Occurrences:N", title="Occurrence").sort('-x')
).properties(
    title="Most Common Other Occurrences Across All Jails"
)

In [16]:
other_occurrence_chart

alt.Chart(...)

In [50]:
top_counties = (
    samples_df
    .select(["Cleaned Facility Name", "Cleaned Occurrences"])
    .with_columns(
        pl.col("Cleaned Occurrences").str.split(",").alias("Cleaned Occurrences")
    )
    .explode("Cleaned Occurrences")
    .with_columns(
        pl.col("Cleaned Occurrences").str.strip_chars().alias("Cleaned Occurrences")
    )
    .filter(pl.col("Cleaned Occurrences").is_not_null())
)

top_counties = top_counties.join(
    top_counties
    .select(pl.col("Cleaned Facility Name").value_counts())
    .unnest("Cleaned Facility Name")  # converts struct to columns
    .rename({"count": "count"}),  # optional, but clarifies
    on="Cleaned Facility Name",
    how="left"
)

def get_county_occurrences(current_jail):
    filtered_df = top_counties.filter(pl.col("Cleaned Facility Name") == current_jail)
    current_jail_result = (
    filtered_df.select(
        pl.col("Cleaned Occurrences").value_counts(sort=True)
    )
    .unnest("Cleaned Occurrences")
    .head(12)
    )
    current_jail_result = current_jail_result.filter(~pl.col("Cleaned Occurrences").is_in(["Error", "no occurrence found"]))
    current_jail_chart = alt.Chart(current_jail_result).mark_bar().encode(
    x=alt.X("count:Q", title="Count"),
    y=alt.Y("Cleaned Occurrences:N", title="Occurrence").sort('-x')
    ).properties(
        title=current_jail
    )
    return current_jail_chart

def get_county_occurrences_percent(current_jail):
    filtered_df = top_counties.filter(pl.col("Cleaned Facility Name") == current_jail)
    current_jail_result = (
        filtered_df.select(
            pl.col("Cleaned Occurrences").value_counts(sort=True)
        )
        .unnest("Cleaned Occurrences")
        .filter(~pl.col("Cleaned Occurrences").is_in(["Error", "no occurrence found"]))
    )
    total_count = current_jail_result["count"].sum()
    current_jail_result = current_jail_result.with_columns(
        (pl.col("count") / total_count * 100).alias("percentage")
    )
    current_jail_result = current_jail_result.sort("count", descending=True).head(10)
    current_jail_chart = alt.Chart(current_jail_result.to_pandas()).mark_bar().encode(
        x=alt.X("percentage:Q", title="Percentage"),
        y=alt.Y("Cleaned Occurrences:N", title="Occurrence").sort('-x')
    ).properties(
        title=current_jail
    )

    return current_jail_chart

cook_county = get_county_occurrences("Cook County Department Of Corrections")
cook_county_percent = get_county_occurrences_percent("Cook County Department Of Corrections")

kane_county = get_county_occurrences("Kane County Sheriffs Office")
kane_county_percent = get_county_occurrences_percent("Kane County Sheriffs Office")

madison_county = get_county_occurrences("Madison County Sheriffs Office")
madison_county_percent = get_county_occurrences_percent("Madison County Sheriffs Office")

lake_county = get_county_occurrences("Lake County Jail")
lake_county_percent = get_county_occurrences_percent("Lake County Jail")

peoria = get_county_occurrences("Peoria County Sheriffs Office")
peoria_percent = get_county_occurrences_percent("Peoria County Sheriffs Office")


In [51]:
cook_county_percent

alt.Chart(...)

In [52]:
kane_county_percent

alt.Chart(...)

In [53]:
madison_county_percent

alt.Chart(...)

In [55]:
lake_county_percent

alt.Chart(...)

In [56]:
peoria_percent

alt.Chart(...)

In [21]:
pattern = r"\bRestraints\b"

restraints_table = top_counties.filter(
    pl.col("Cleaned Occurrences").str.contains(pattern)
)

restraints_table = (
    restraints_table.select(
        pl.col("Cleaned Facility Name").value_counts(sort=True)
    )
    .unnest("Cleaned Facility Name")
    .head(21)
)

restraints_table = restraints_table.filter(pl.col("Cleaned Facility Name") != "")

restraints_chart = alt.Chart(restraints_table).mark_bar().encode(
    x=alt.X("count:Q", title="Count"),
    y=alt.Y("Cleaned Facility Name:N", title="Jail").sort('-x')
).properties(
    title="Top Jails for Restraint Usage"
)

# Footnote as a separate text chart
footnote = alt.Chart(pl.DataFrame({
    "text": ["Note: Missing 214 Jail Names (12%)"],
    "x": [0],
    "y": [0]
})).mark_text(
    align="left",
    baseline="bottom",
    fontSize=10,
    dx=0,
    dy=20,  # distance below the main chart area
    color="gray"
).encode(
    x=alt.value(0),
    y=alt.value(430),  # just below the chart (chart height + some offset)
    text="text:N"
)

# Use layering to draw the footnote over/under the chart
restraints_chart = alt.layer(
    restraints_chart,
    footnote
).configure_view(
    stroke=None  # removes box around the chart
)

In [ ]:
restraints_chart

alt.LayerChart(...)

In [23]:
restraints_table = restraints_table.filter(pl.col("Cleaned Facility Name") != "null").join(top_counties, on="Cleaned Facility Name")

In [24]:
restraints_table = restraints_table.filter(
    pl.col("Cleaned Occurrences") == "Restraints Used"
)

restraints_table = restraints_table.unique()

In [25]:
restraints_perc_chart = alt.Chart(restraints_table.filter(pl.col("count_right") >= 100).with_columns(
    ((pl.col("count") / pl.col("count_right")).alias("perc")*100)
)).mark_bar().encode(
    x=alt.X("perc:Q", title="Percentage"),
    y=alt.Y("Cleaned Facility Name:N", title="Jail").sort('-x')
).properties(
    title="Top Jails for Restraint Usage as a Percent of Reports"
)

# Footnote as a separate text chart
footnote = alt.Chart(pl.DataFrame({
    "text": ["Note: Only Jails with over 100 Reports Total"],
    "x": [0],
    "y": [0]
})).mark_text(
    align="left",
    baseline="bottom",
    fontSize=10,
    dx=0,
    dy=20,  # distance below the main chart area
    color="gray"
).encode(
    x=alt.value(0),
    y=alt.value(440),  # just below the chart (chart height + some offset)
    text="text:N"
)

# Use layering to draw the footnote over/under the chart
restraints_perc_chart = alt.layer(
    restraints_perc_chart,
    footnote
).configure_view(
    stroke=None  # removes box around the chart
)

In [26]:
restraints_perc_chart

alt.LayerChart(...)

In [27]:
pattern = r"\bFighting among Detainees\b"

fighting_table = top_counties.filter(
    pl.col("Cleaned Occurrences").str.contains(pattern)
)

fighting_table = (
    fighting_table.select(
        pl.col("Cleaned Facility Name").value_counts(sort=True)
    )
    .unnest("Cleaned Facility Name")
    .head(11)
)

fighting_table_table = fighting_table.filter(pl.col("Cleaned Facility Name") != "")

fighting_chart = alt.Chart(fighting_table).mark_bar().encode(
    x=alt.X("count:Q", title="Count"),
    y=alt.Y("Cleaned Facility Name:N", title="Jail").sort('-x')
).properties(
    title="Top Jails for Fighting among Detainees"
)

# Footnote as a separate text chart
footnote = alt.Chart(pl.DataFrame({
    "text": ["Note: Missing 214 Jail Names (12%)"],
    "x": [0],
    "y": [0]
})).mark_text(
    align="left",
    baseline="bottom",
    fontSize=10,
    dx=0,
    dy=20,  # distance below the main chart area
    color="gray"
).encode(
    x=alt.value(0),
    y=alt.value(250),  # just below the chart (chart height + some offset)
    text="text:N"
)

# Use layering to draw the footnote over/under the chart
fighting_chart = alt.layer(
    fighting_chart,
    footnote
).configure_view(
    stroke=None  # removes box around the chart
)

In [28]:
fighting_table

Cleaned Facility Name,count
str,u32
"""Cook County Department Of Corr…",4768
"""Madison County Sheriffs Office""",445
"""Kane County Sheriffs Office""",170
"""Sangamon County Jail""",110
"""Lake County Jail""",103
…,…
"""Winnebago County Jail""",75
"""Dupage County Jail""",59
"""Sangamon County Adult Detentio…",43


In [29]:
fighting_chart

alt.LayerChart(...)

In [30]:
fighting_table = fighting_table.filter(pl.col("Cleaned Facility Name") != "null").join(top_counties, on="Cleaned Facility Name")

In [31]:
fighting_table = fighting_table.filter(
    pl.col("Cleaned Occurrences") == "Restraints Used"
)

fighting_table = fighting_table.unique()

fighting_perc_chart = alt.Chart(fighting_table.filter(pl.col("count_right") >= 100).with_columns(
    ((pl.col("count") / pl.col("count_right")).alias("perc")*100)
)).mark_bar().encode(
    x=alt.X("perc:Q", title="Percentage"),
    y=alt.Y("Cleaned Facility Name:N", title="Jail").sort('-x')
).properties(
    title="Top Jails for Restraint Usage as a Percent of Reports"
)

# Footnote as a separate text chart
footnote = alt.Chart(pl.DataFrame({
    "text": ["Note: Only Jails with over 100 Reports Total"],
    "x": [0],
    "y": [0]
})).mark_text(
    align="left",
    baseline="bottom",
    fontSize=10,
    dx=0,
    dy=20,  # distance below the main chart area
    color="gray"
).encode(
    x=alt.value(0),
    y=alt.value(230),  # just below the chart (chart height + some offset)
    text="text:N"
)

# Use layering to draw the footnote over/under the chart
fighting_perc_chart = alt.layer(
    fighting_perc_chart,
    footnote
).configure_view(
    stroke=None  # removes box around the chart
)

In [32]:
fighting_perc_chart

alt.LayerChart(...)

In [38]:
top_people = (
    samples_df.select(
        pl.col("Inmate Names").value_counts(sort=True)
    )
    .unnest("Inmate Names")
    .head(10)
    )

case_study = samples_df.filter(
    pl.col("Inmate Names") == " Ordonez Robert F"
)

In [ ]:
case_study

Inmate Names,count
str,u32
"""""",4633
""" Ordonez Robert F""",51
"""Laveau Christopher T""",29
"""Clark Keith""",29
"""Leo McGuinn""",25
""" Bradley Deandre""",25
"""Travis Williams""",24
"""Fenderson Amos""",21
"""Carlos Cruz""",21


In [33]:
#samples_df.write_excel("7_10_sample.xlsx")

To-Do:
- Breakdown of largest counties